# Paired protein–PTM BMI analysis

Bivariate Student-t models for parent protein and phospho/glyco features, including draw-wise protein-conditioned effects, numerical diagnostics, sensitivity fits and influence refits. Configure paths, shard ownership and execution mode in the first code cell. Run all four worker shards before aggregation. The published run contract and its legacy fingerprint validator are retained for checkpoint compatibility.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, zipfile, time, traceback, types, importlib.metadata
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path('/content/drive/MyDrive/new_dysfunction')
INPUT_ZIP=PROJECT_ROOT/'multiomics_inputs/PDAC_main_cohort_multiomics_inputs.zip'
SIGNATURE_INPUT_ZIP=PROJECT_ROOT/'multiomics_inputs/PDAC_original_signature_inputs.zip'
OUTPUT_BASE=PROJECT_ROOT/'multiomics_extensions/protein_ptm_paired_v2'
WORK=Path('/content/pdac_ptm_paired_v2');WORK.mkdir(parents=True,exist_ok=True)
# Published run root shared by shards 0–3.
RESUME_RUN_ROOT=Path('/content/drive/MyDrive/new_dysfunction/multiomics_extensions/protein_ptm_paired_v2/publication/2687095c8b4e1e44/shards_4')
# None enables compatible-run discovery or initializes a new run.
MODE='worker'                  # preflight, worker, aggregate
PROFILE='publication'              # publication, smoke
N_SHARDS=4
SHARD_ID=0                        # four sessions: 0, 1, 2, 3
ASSAYS=['phospho','glyco']
MIN_PAIRED_N=102
MIN_PLEXES=16
FEATURE_SCOPE='original_signatures_plus_targets'  # or all_measured
# Optional CSV: assay, feature_id, pass_qc (true/false), provenance.
# Exact IDs and no duplicates. Missing features fail QC when a file is provided.
SITE_QC_CSV=None
# No log-ratio output unless documented compatible log2 quantities are verified.
COMPATIBLE_LOG2_RATIO=False
SCALE_PROVENANCE_NOTE=''
SENSITIVITY_TARGETS={'phospho':['NP_644805.1_Y705','NP_644805.1_S727'], 'glyco_genes':['ASAH1']}
RUN_SENSITIVITIES=True
RUN_CATEGORICAL=True               # named targets only; normal weight reference
RUN_INFLUENCE_REFITS=True          # all primary PSIS-flagged pairs
BOOTSTRAP_REPLICATES=2000          # named targets; same within-plex patient resamples
assert MODE in ('preflight','worker','aggregate') and PROFILE in ('smoke','publication')
if PROFILE=='smoke':N_SHARDS,SHARD_ID=1,0  # one session exercises both assays
assert 0<=SHARD_ID<N_SHARDS and N_SHARDS>0
assert set(ASSAYS)<= {'phospho','glyco'} and len(set(ASSAYS))==len(ASSAYS) and ASSAYS
assert FEATURE_SCOPE in ('original_signatures_plus_targets','all_measured')
assert 3<=MIN_PAIRED_N<=127 and 2<=MIN_PLEXES<=24
if COMPATIBLE_LOG2_RATIO and not SCALE_PROVENANCE_NOTE.strip():
    raise ValueError('Document compatible log2 scales before enabling raw log-ratio contrasts.')


In [ ]:
import subprocess
critical = ['numpy', 'scipy', 'pandas', 'pymc', 'arviz', 'pytensor', 'xarray', 'h5py']
loaded_before = {p: getattr(sys.modules[p], '__version__', None) for p in critical if p in sys.modules}
requirements = [
    'pymc==5.25.1', 'arviz==0.22.0', 'numpy>=2.0,<2.3',
    'scipy>=1.11,<1.16', 'pandas>=2.2,<3', 'xarray>=2025.1,<2026',
    'matplotlib>=3.8,<4', 'h5netcdf>=1.3,<2', 'h5py>=3.10,<4',
]
# Keep the existing run's recorded package versions, rather than silently moving folders.
if RESUME_RUN_ROOT is not None:
    saved=sorted(Path(RESUME_RUN_ROOT).glob('*/contract.json'))
    if not saved:raise FileNotFoundError(f'No saved contract in {RESUME_RUN_ROOT}; check the mounted Drive and resume path.')
    resume_contract=json.loads(saved[0].read_text())
    pins=resume_contract['versions']
    requirements=[r for r in requirements if r.split('=')[0].split('>')[0].split('<')[0] not in pins]
    requirements += [f'{package}=={version}' for package,version in pins.items()]
    print('Restoring package versions recorded by the existing run:',pins)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--disable-pip-version-check', '--upgrade-strategy', 'only-if-needed',
                *requirements], check=True)
changed = [p for p, v in loaded_before.items() if v and v != importlib.metadata.version(p)]
if changed:
    raise RuntimeError('Packages updated after import: ' + ', '.join(changed) +
                       '. Choose Runtime → Restart session, then Run all again.')
print('Colab Python:', sys.version.split()[0], '| sampler: PyMC NUTS')


In [ ]:
os.environ['PYTENSOR_FLAGS']='blas__ldflags='
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ[name]='1'
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from scipy.linalg import helmert
from threadpoolctl import threadpool_limits
threadpool_limits(1)
LAYERS=['Protein','PTM']
ANALYSIS_VERSION='protein_ptm_paired_v2.0.0'
EXPECTED_SIGNATURE_ZIP_SHA256='8098755eeca1d564a4e5e004aa9bdfd0e16f187b703f21110fe517c45200eeab'
profile=({'CHAINS':4,'TUNE':2000,'DRAWS':4000,'PPC_DRAWS':1000} if PROFILE=='publication'
         else {'CHAINS':2,'TUNE':150,'DRAWS':150,'PPC_DRAWS':100})
CONFIG=dict(PROFILE=PROFILE,SEED=20260921,SAMPLER='pymc',CORES=2,TARGET_ACCEPT=.99,MAX_TREEDEPTH=12,
    BMI_PRIOR_SD=.5,COVARIATE_PRIOR_SD=.5,GROUP_SD_PRIOR=.3,FIXED_GROUP_PRIOR_SD=2.,
    RETRY_ONCE=PROFILE=='publication',**profile)
print('No ROPE or FDR selection. Report posterior means and pointwise 95% HDIs.')


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for part in iter(lambda: handle.read(2**20), b''):
            h.update(part)
    return h.hexdigest()

def canonical_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, allow_nan=False,
                                    separators=(',', ':')).encode()).hexdigest()

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.partial')
    temp.write_text(json.dumps(obj, indent=2, allow_nan=False, default=str))
    os.replace(temp, path)

def write_csv(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.partial')
    pd.DataFrame(data).to_csv(temp, index=False)
    os.replace(temp, path)

def truth(series):
    return series.astype(str).str.strip().str.lower().isin(['true', '1', 'yes'])

In [ ]:
def unpack_inputs(path,work):
    if not path.is_file(): raise FileNotFoundError(path)
    digest=sha256_file(path); dest=work/('inputs_'+digest[:16]);dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as z:
        names=z.namelist()
        manifests=[s for s in names if s.endswith('/package_manifest.json')]
        if len(manifests)!=1: raise ValueError('Expected one package manifest')
        prefix=manifests[0].removesuffix('package_manifest.json')
        manifest=json.loads(z.read(manifests[0]))
        for item in manifest:
            rel=item['path']; target=(dest/rel).resolve()
            if not target.is_relative_to(dest.resolve()): raise ValueError('Unsafe archive path')
            content=z.read(prefix+rel)
            if len(content)!=item['bytes'] or hashlib.sha256(content).hexdigest()!=item['sha256']:
                raise ValueError('Input checksum failure: '+rel)
            target.parent.mkdir(parents=True,exist_ok=True)
            if not target.exists() or sha256_file(target)!=item['sha256']: target.write_bytes(content)
    return dest,digest

def load_signature_links(path,input_hash):
    if not path.is_file():
        if MODE=='preflight':return pd.DataFrame(),None,{'status':'missing original-signature ZIP; full fits blocked'}
        raise FileNotFoundError(f'Use the same signature ZIP as your RNA–protein notebook: {path}')
    if sha256_file(path)!=EXPECTED_SIGNATURE_ZIP_SHA256: raise ValueError('Original signature ZIP changed')
    with zipfile.ZipFile(path) as z:
        payload=json.loads(z.read('payload.json'))
        if payload['input_zip_sha256']!=input_hash: raise ValueError('Cohort ZIP differs from archived original run')
        content=z.read(payload['signature_file'])
        if hashlib.sha256(content).hexdigest()!=payload['signature_sha256']: raise ValueError('Signature checksum mismatch')
        signatures=json.loads(content)
    inv=pd.DataFrame(payload['original_features'])
    if len(inv)!=914 or not inv.global_feature.is_unique: raise ValueError('Expected 914 original labels')
    rows=[]
    for f in inv.to_dict('records'):
        raw=signatures[f['CellType']][f['Signature']]
        if not isinstance(raw,list) or not all(isinstance(g,str) for g in raw): raise ValueError('Unsigned original schema changed')
        genes=sorted(set(g.strip().upper() for g in raw if g.strip()))
        h=hashlib.sha256(('POS:'+','.join(genes)+'|NEG:').encode()).hexdigest()
        if h!=f['source_gene_hash']: raise ValueError('Source gene list hash mismatch')
        for gene in genes: rows.append({k:f[k] for k in ['global_feature','family','CellType','Signature','source_gene_hash']}|{'gene':gene})
    return pd.DataFrame(rows),payload,{'status':'verified','original_labels':914}

def load_data(root,payload):
    p=root/'prepared'; cohort=pd.read_csv(p/'model_covariates.csv')
    ids=cohort.base_sample_id.astype(str).tolist()
    ref=pd.read_csv(p/'main_cohort_127.csv')
    assert len(ids)==len(set(ids))==127 and ids==ref.base_sample_id.tolist()
    assert np.isfinite(cohort[['bmi','age','sex_centered']]).all().all()
    assert cohort.bmi_group.value_counts().to_dict()=={'overweight':57,'normal_weight':52,'obese':18}
    contract=json.loads((p/'preparation_contract.json').read_text())
    assert sha256_file(root/'provenance/analysis_cohort.csv')==contract['main_cohort_sha256']
    for col in ['adenosquamous_recorded','weight_loss_recorded']:cohort[col]=truth(cohort[col])
    if payload is not None:
        purity=pd.DataFrame(payload['dna_purity_records'])
        cohort=cohort.merge(purity,on='base_sample_id',how='left',validate='one_to_one',sort=False)
        for col in ['wes_purity','wgs_purity']:
            assert cohort[col].dropna().between(0,1).all()
    mutation=pd.read_csv(p/'matrices/mutation_tumor_main_cohort.tsv.gz',sep='\t',index_col=0,keep_default_na=False)
    nonsyn={'missense_variant','frameshift_variant','stop_gained','stop_lost','start_lost',
        'inframe_deletion','inframe_insertion','splice_donor_variant','splice_acceptor_variant'}
    import re
    for gene in ['TP53','CDKN2A','SMAD4']:
        calls=mutation.loc[gene,ids]
        if calls.eq('').any(): raise ValueError('Missing mutation status')
        cohort['mutation_'+gene]=[int(bool(set(re.split('[,;]',s))&nonsyn)) for s in calls]
    plex=pd.read_csv(p/'tumor_plex_map.csv')
    assert not plex.duplicated(['layer','base_sample_id']).any()
    by_layer={}
    for layer in ['protein']+ASSAYS:
        s=plex[plex.layer.eq(layer)].set_index('base_sample_id').loc[ids]
        assert len(s)==127 and s.plex.notna().all()
        by_layer[layer]=s.plex.to_numpy()
    # Current source has identical patient batch partitions across assays, even when names differ.
    for layer in ASSAYS:
        a=by_layer[layer]; b=by_layer['protein']
        if not np.array_equal(a[:,None]==a[None,:],b[:,None]==b[None,:]):
            raise ValueError('Assay batch partitions differ: needs an explicitly separate-batch model')
    matrices={}; annotations={}
    for layer in ['protein']+ASSAYS:
        with np.load(p/f'matrices/{layer}_tumor_main_cohort.npz',allow_pickle=False) as z:
            assert z['patient_ids'].tolist()==ids
            values=z['values'].astype(float);features=z['feature_ids'].astype(str)
        assert values.shape==(len(features),127) and len(set(features))==len(features)
        assert not np.isinf(values).any()
        ann=pd.read_csv(p/f'feature_metadata/{layer}_annotations.csv',keep_default_na=False)
        assert ann.feature_id.tolist()==features.tolist()
        matrices[layer]={'values':values,'feature_ids':features}
        annotations[layer]=ann
    return dict(cohort=cohort,matrices=matrices,annotations=annotations,batches=by_layer,contract=contract)


In [ ]:
def make_design(cohort,mask,adjustment='mundlak',extra=None,categorical=False):
    d=cohort.loc[mask].copy().reset_index(drop=True)
    levels=sorted(d.plex.unique());group=pd.Categorical(d.plex,categories=levels).codes
    raw=({'overweight':d.bmi_group.eq('overweight').to_numpy(float),'obese':d.bmi_group.eq('obese').to_numpy(float)}
         if categorical else {'bmi_5':d.bmi.to_numpy()/5})
    focal=list(raw)
    raw.update(age_10y=d.age.to_numpy()/10,sex=d.sex_centered.to_numpy())
    if extra=='stage':
        for stage in ['II','III','IV']:raw['stage_'+stage]=d.stage_major.eq(stage).to_numpy(float)
    elif extra=='mutations':
        for gene in ['TP53','CDKN2A','SMAD4']:raw['mutation_'+gene]=d['mutation_'+gene].to_numpy(float)
    elif extra=='wes_purity':raw['wes_purity_10pct']=d.wes_purity.to_numpy()/0.1
    names=[];columns=[];between=[];dropped=[]
    for key,x in raw.items():
        if not np.isfinite(x).all():raise ValueError('Non-finite covariate '+key)
        mean=pd.Series(x).groupby(group).transform('mean').to_numpy();v=x-mean
        if v.std()<1e-10:
            if key in focal:raise ValueError('No within-plex variation for '+key)
            dropped.append(key+'_within')
        else:names.append(key+'_within');columns.append(v)
        if adjustment=='mundlak':between.append((key+'_between',mean-x.mean()))
    for name,value in between:
        if value.std()<1e-10:dropped.append(name)
        else:names.append(name);columns.append(value)
    X=np.column_stack(columns)
    Q=helmert(len(levels),full=False).T if adjustment=='fixed' else None
    full=np.column_stack([np.ones(len(d)),X]+([Q[group]] if Q is not None else []))
    if np.linalg.matrix_rank(full)!=full.shape[1]:raise ValueError('Rank-deficient design; no confounder deletion')
    return dict(data=d,X=X,names=names,group=group,levels=levels,Q=Q,adjustment=adjustment,extra=extra,
        dropped_constant_terms=dropped,focal=[s+'_within' for s in focal],categorical=categorical)


In [ ]:
def prepare_features(data,links,qc_path=None):
    protein=data['matrices']['protein'];lookup={g:i for i,g in enumerate(protein['feature_ids'])}
    genes=set(links.gene) if len(links) else set();audits=[];records=[];arrays={}
    qc=None
    if qc_path:
        qc=pd.read_csv(qc_path,keep_default_na=False)
        if not {'assay','feature_id','pass_qc','provenance'}<=set(qc):raise ValueError('Invalid QC schema')
        if qc.duplicated(['assay','feature_id']).any():raise ValueError('Duplicate QC feature')
        if qc.provenance.eq('').any():raise ValueError('QC provenance missing')
        qc={(r.assay,r.feature_id):str(r.pass_qc).lower() in ('true','1','yes') for r in qc.itertuples()}
    for assay in ASSAYS:
        matrix=data['matrices'][assay];ann=data['annotations'][assay]
        for row in ann.itertuples(index=True):
            sid=row.feature_id;gene=row.Gene.strip().upper();reason='';n=0;nplex=0
            target=(sid in SENSITIVITY_TARGETS['phospho'] if assay=='phospho' else gene in SENSITIVITY_TARGETS['glyco_genes'])
            if FEATURE_SCOPE=='original_signatures_plus_targets' and gene not in genes and not target:
                reason='outside original genes and named targets' if len(links) else 'signature archive unavailable; scope unresolved'
            elif qc is not None and not qc.get((assay,sid),False):reason='external QC not passed or absent'
            elif gene not in lookup:reason='parent protein unavailable by exact gene symbol'
            else:
                pair=np.column_stack([protein['values'][lookup[gene]],matrix['values'][row.Index]])
                mask=np.isfinite(pair).all(1);n=int(mask.sum());nplex=len(set(data['batches'][assay][mask]))
                if n<MIN_PAIRED_N or nplex<MIN_PLEXES:reason='paired coverage below threshold'
                elif np.any(pair[mask].std(0,ddof=1)<1e-8):reason='constant paired response'
            rec=dict(assay=assay,feature_id=sid,gene=gene,n_paired=n,n_plexes=nplex,target=target,
                included=not bool(reason),exclusion_reason=reason,localization_policy='external_QC' if qc is not None else 'source_processed_unverified_localization')
            audits.append(rec)
            if reason:continue
            mean=pair[mask].mean(0);sd=pair[mask].std(0,ddof=1);Y=(pair-mean)/sd
            key=assay+'_'+hashlib.sha256(sid.encode()).hexdigest()[:20]
            arrays[key]={'Y':Y,'mask':mask,'means':mean,'sds':sd}
            records.append(rec|dict(key=key,protein_row=gene,protein_mean=float(mean[0]),protein_sd=float(sd[0]),
                ptm_mean=float(mean[1]),ptm_sd=float(sd[1]),parent_resolution='gene_level'))
    return pd.DataFrame(records),pd.DataFrame(audits),arrays

def feature_cohort(data,assay):
    d=data['cohort'].copy();d['plex']=data['batches'][assay];return d

def variant_spec(d,base,variant):
    mask=base.copy();extra=None;adjustment='mundlak';categorical=False
    if variant=='fixed_plex':adjustment='fixed'
    elif variant in ('stage_subset_base','stage_adjusted'):
        mask &= d.stage_major.isin(['I','II','III','IV']).to_numpy()
        if variant=='stage_adjusted':extra='stage'
    elif variant in ('wes_subset_base','wes_adjusted'):
        if 'wes_purity' not in d:raise ValueError('WES purity absent from original payload')
        mask &= np.isfinite(d.wes_purity.to_numpy())
        if variant=='wes_adjusted':extra='wes_purity'
    elif variant=='mutation_adjusted':extra='mutations'
    elif variant=='exclude_recorded_weight_loss':mask &= ~d.weight_loss_recorded.to_numpy(bool)
    elif variant=='exclude_adenosquamous':mask &= ~d.adenosquamous_recorded.to_numpy(bool)
    elif variant=='categorical':categorical=True
    elif variant!='primary':raise ValueError(variant)
    if mask.sum()<30:raise ValueError('Fewer than 30 patients in sensitivity subset')
    return mask,adjustment,extra,categorical

def plan_jobs(data,features,arrays):
    jobs=[];excluded=[]
    variants=['fixed_plex','stage_subset_base','stage_adjusted','wes_subset_base','wes_adjusted',
        'mutation_adjusted','exclude_recorded_weight_loss','exclude_adenosquamous']
    if RUN_CATEGORICAL:variants+=['categorical']
    for f in features.to_dict('records'):
        d=feature_cohort(data,f['assay'])
        for variant in ['primary']+(variants if f['target'] and RUN_SENSITIVITIES and PROFILE!='smoke' else []):
            try:
                mask,adj,extra,cat=variant_spec(d,arrays[f['key']]['mask'],variant)
                des=make_design(d,mask,adj,extra,cat)
                jobs.append(dict(key=f['key'],assay=f['assay'],variant=variant,n_patients=int(mask.sum()),
                    n_plexes=len(des['levels']),shard=int(hashlib.sha256(f['key'].encode()).hexdigest()[:12],16)%N_SHARDS))
            except ValueError as exc:
                excluded.append(dict(key=f['key'],variant=variant,reason=str(exc)))
                if variant=='primary':break
    return pd.DataFrame(jobs),pd.DataFrame(excluded)


In [ ]:
ROOT,INPUT_HASH=unpack_inputs(INPUT_ZIP,WORK)
LINKS,PAYLOAD,SIGNATURE_STATUS=load_signature_links(SIGNATURE_INPUT_ZIP,INPUT_HASH)
DATA=load_data(ROOT,PAYLOAD)
FEATURES,ELIGIBILITY,ARRAYS=prepare_features(DATA,LINKS,SITE_QC_CSV)
if FEATURES.empty:raise ValueError('No eligible features. Resolve inputs and thresholds before fitting.')
if PROFILE=='smoke':
    # Prefer named biology, then use lexical feature IDs. Never select by BMI effects.
    FEATURES=FEATURES.sort_values(['assay','target','feature_id'],ascending=[True,False,True]).groupby('assay',sort=True).head(1).copy()
JOBS,UNIDENTIFIABLE=plan_jobs(DATA,FEATURES,ARRAYS)
if MODE!='preflight' and (PAYLOAD is None or JOBS.empty):raise ValueError('Resolve signature inputs and estimability before fitting')
print('Signature status:',SIGNATURE_STATUS)
print('Coverage:',FEATURES.groupby('assay').size().to_dict())
print('Planned fits:',len(JOBS),' | unidentifiable variants:',len(UNIDENTIFIABLE))
display(JOBS.groupby(['assay','variant']).size().rename('fits').reset_index())
display(FEATURES[FEATURES.target][['assay','feature_id','gene','n_paired','n_plexes']])
print('Full site analysis can require thousands of fits; four shards divide this workload, not its total size.')


In [ ]:
def build_model(design, observed, config):
    import pymc as pm
    import pytensor.tensor as pt
    coords = {'patient': design['data'].base_sample_id.tolist(), 'layer': LAYERS,
              'term': design['names'], 'plex': design['levels']}
    if design['adjustment'] == 'fixed':
        coords['plex_contrast'] = list(range(len(design['levels'])-1))
    prior = np.full(len(design['names']), config['COVARIATE_PRIOR_SD'])
    for i, name in enumerate(design['names']):
        if name.startswith('bmi_5_'): prior[i] = config['BMI_PRIOR_SD']
    with pm.Model(coords=coords) as model:
        alpha = pm.Normal('alpha', 0, 0.5, dims='layer')
        coefficient = pm.Normal('coefficient', 0, prior[:, None], dims=('term', 'layer'))
        if design['adjustment'] == 'mundlak':
            group_scale = pm.HalfNormal('group_scale', config['GROUP_SD_PRIOR'], dims='layer')
            group_z = pm.Normal('group_z', 0, 1, dims=('plex', 'layer'))
            group_effect = pm.Deterministic('group_effect', group_z*group_scale,
                                           dims=('plex', 'layer'))
        else:
            group_coefficient = pm.Normal('group_coefficient', 0,
                                           config['FIXED_GROUP_PRIOR_SD'],
                                           dims=('plex_contrast', 'layer'))
            group_effect = pm.Deterministic('group_effect',
                pt.dot(design['Q'], group_coefficient), dims=('plex', 'layer'))
        mu = alpha + pt.dot(design['X'], coefficient) + group_effect[design['group']]
        chol, corr, scales = pm.LKJCholeskyCov('resid_chol', n=2, eta=2,
            sd_dist=pm.HalfNormal.dist(1.0, shape=2), compute_corr=True)
        nu_minus_two = pm.Exponential('nu_minus_two', 0.1)
        nu = pm.Deterministic('nu', 2+nu_minus_two)
        positions=[design['names'].index(s) for s in design['focal']]
        slopes=coefficient[positions]
        coupling=pm.Deterministic('protein_to_ptm_coupling',corr[0,1]*scales[1]/scales[0])
        pm.Deterministic('focal_effect',slopes)
        pm.Deterministic('ptm_minus_protein',slopes[:,1]-slopes[:,0])
        pm.Deterministic('ptm_conditional_effect',slopes[:,1]-coupling*slopes[:,0])
        pm.Deterministic('residual_correlation',corr[0,1])
        pm.MvStudentT('paired_score', nu=nu, mu=mu, chol=chol,
                      observed=observed, dims=('patient', 'layer'))
    return model

In [ ]:
def posterior_array(idata, name):
    da = idata.posterior[name].stack(sample=('chain', 'draw'))
    return da.transpose('sample', *[d for d in da.dims if d != 'sample']).values

def hdi(values, probability=0.95):
    a = np.sort(np.asarray(values, float).ravel())
    if not np.isfinite(a).all() or len(a) < 20:
        raise ValueError('Cannot summarize incomplete posterior draws')
    width = int(np.floor(probability*len(a)))
    left = int(np.argmin(a[width:]-a[:-width]))
    return float(a[left]), float(a[left+width])

def numerical_diagnostics(idata, model, config, folder):
    import arviz as az
    names = [v.name for v in model.free_RVs] + ['focal_effect','ptm_conditional_effect','protein_to_ptm_coupling']
    summary = az.summary(idata, var_names=names, kind='diagnostics', round_to='none')
    summary.rename_axis('parameter').reset_index().to_csv(folder/'parameter_diagnostics.csv', index=False)
    columns = summary[['r_hat', 'ess_bulk', 'ess_tail']].to_numpy(float)
    finite = bool(np.isfinite(columns).all())
    rhat = float(np.nanmax(summary.r_hat))
    bulk, tail = float(np.nanmin(summary.ess_bulk)), float(np.nanmin(summary.ess_tail))
    stats = idata.sample_stats
    divergences = int(stats['diverging'].sum())
    bfmi = float(np.nanmin(az.bfmi(idata)))
    if 'reached_max_treedepth' in stats:
        depth_hits = int(stats['reached_max_treedepth'].sum())
    elif 'tree_depth' in stats:
        depth_hits = int((stats.tree_depth >= config['MAX_TREEDEPTH']).sum())
    elif 'depth' in stats:
        depth_hits = int((stats.depth >= config['MAX_TREEDEPTH']).sum())
    else:
        raise RuntimeError('Sampler returned no tree-depth statistic; cannot verify the gate')
    passed = finite and rhat <= 1.01 and min(bulk, tail) >= 400 and bfmi >= 0.30 \
             and divergences == 0 and depth_hits == 0
    return {'max_rhat': rhat if np.isfinite(rhat) else None,
            'min_ess_bulk': bulk if np.isfinite(bulk) else None,
            'min_ess_tail': tail if np.isfinite(tail) else None,
            'min_bfmi': bfmi if np.isfinite(bfmi) else None,
            'divergences': divergences, 'maximum_depth_hits': depth_hits,
            'all_diagnostics_finite': finite, 'numerical_checks_pass': bool(passed),
            'publication_profile': config['PROFILE'] == 'publication'}

In [ ]:
def effect_summary(idata,feature,variant,design,passed):
    beta=posterior_array(idata,'focal_effect');conditional=posterior_array(idata,'ptm_conditional_effect')
    rows=[]
    def add(quantity,draws,term,units):
        lo,hi=hdi(draws)
        rows.append(dict(key=feature['key'],assay=feature['assay'],feature_id=feature['feature_id'],gene=feature['gene'],
            variant=variant,term=term,quantity=quantity,posterior_mean=float(np.mean(draws)),
            hdi_95_lower=lo,hdi_95_upper=hi,hdi_excludes_zero=bool(lo>0 or hi<0),
            probability_positive=float(np.mean(draws>0)),n_patients=len(design['data']),n_plexes=len(design['levels']),
            numerical_checks_pass=bool(passed),effect_units=units,interval_type='pointwise 95% HDI; no multiplicity control'))
    for j,term in enumerate(design['focal']):
        units='primary-paired-sample response SD'+(' vs normal weight, within plex' if design['categorical'] else ' per 5 kg/m2, within plex')
        for label,values in [('Protein',beta[:,j,0]),('PTM',beta[:,j,1]),
            ('PTM_minus_Protein_standardized',beta[:,j,1]-beta[:,j,0]),('PTM_given_Protein',conditional[:,j])]:
            add(label,values,term,units)
        if COMPATIBLE_LOG2_RATIO:
            add('log2_PTM_minus_Protein',feature['ptm_sd']*beta[:,j,1]-feature['protein_sd']*beta[:,j,0],term,
                'log2 ratio'+(' vs normal weight' if design['categorical'] else ' per 5 kg/m2'))
    add('residual_Protein_PTM_correlation',posterior_array(idata,'residual_correlation'),'residual','correlation')
    add('protein_to_ptm_coupling',posterior_array(idata,'protein_to_ptm_coupling'),'residual','PTM SD per protein SD')
    return pd.DataFrame(rows)


In [ ]:
def compute_ppc(idata, design, observed, config, folder, seed):
    rng = np.random.default_rng(seed+231)
    a, c, g = [posterior_array(idata, n) for n in ['alpha', 'coefficient', 'group_effect']]
    total = len(a)
    chosen = np.sort(rng.choice(total, min(config['PPC_DRAWS'], total), replace=False))
    mu = a[chosen,None,:] + np.einsum('nt,stl->snl', design['X'], c[chosen]) \
         + g[chosen][:, design['group'], :]
    scales = posterior_array(idata, 'resid_chol_stds')[chosen]
    corr = posterior_array(idata, 'resid_chol_corr')[chosen]
    nu = posterior_array(idata, 'nu')[chosen]
    S = corr*scales[:,:,None]*scales[:,None,:]
    L = np.linalg.cholesky(S)
    z = rng.normal(size=mu.shape)
    gaussian = np.einsum('sij,snj->sni', L, z)
    w = np.sqrt(rng.chisquare(nu[:,None], size=mu.shape[:2])/nu[:,None])
    replicate = mu + gaussian/w[:,:,None]
    observed_residual = observed[None,:,:]-mu
    replicate_residual = replicate-mu
    checks = []
    def add(name, obs, rep):
        upper = float(np.mean(rep >= obs))
        checks.append({'check': name, 'observed_draw_median': float(np.median(obs)),
                       'replicated_median': float(np.median(rep)),
                       'replicated_95_lower': float(np.quantile(rep,.025)),
                       'replicated_95_upper': float(np.quantile(rep,.975)),
                       'posterior_predictive_upper_tail_probability': upper,
                       'flag': bool(upper < .025 or upper > .975),
                       'interpretation': 'Descriptive predictive check; not a calibrated p-value'})
    for j, layer in enumerate(LAYERS):
        add(layer+'_residual_SD', observed_residual[:,:,j].std(1,ddof=1),
            replicate_residual[:,:,j].std(1,ddof=1))
        add(layer+'_maximum_absolute_residual', np.abs(observed_residual[:,:,j]).max(1),
            np.abs(replicate_residual[:,:,j]).max(1))
    def corrs(x):
        v=x-x.mean(1,keepdims=True)
        return np.sum(v[:,:,0]*v[:,:,1],axis=1)/np.sqrt(
            np.sum(v[:,:,0]**2,axis=1)*np.sum(v[:,:,1]**2,axis=1))
    add('Protein_PTM_residual_correlation', corrs(observed_residual), corrs(replicate_residual))
    low, high = np.quantile(replicate,[.05,.95],axis=0)
    for j, layer in enumerate(LAYERS):
        checks.append({'check': layer+'_90pct_in_sample_predictive_coverage',
                       'observed_draw_median': float(np.mean((observed[:,j]>=low[:,j]) &
                                                            (observed[:,j]<=high[:,j]))),
                       'interpretation': 'In-sample descriptive coverage; not out-of-sample validation',
                       'flag': False})
    write_csv(folder/'predictive_checks.csv', checks)
    write_csv(folder/'patient_residuals.csv', pd.DataFrame({
        'base_sample_id': design['data'].base_sample_id,
        'Protein_residual': observed[:,0]-mu[:,:,0].mean(0),
        'PTM_residual': observed[:,1]-mu[:,:,1].mean(0)}))
    return int(sum(bool(x.get('flag',False)) for x in checks))

def compute_loo(idata, design, folder):
    import arviz as az
    ll = idata.log_likelihood['paired_score']
    point_dims = [d for d in ll.dims if d not in ['chain', 'draw']]
    if len(point_dims)!=1 or ll.sizes[point_dims[0]]!=len(design['data']):
        raise RuntimeError('LOO must have one joint protein/PTM likelihood per patient.')
    loo = az.loo(idata, pointwise=True, var_name='paired_score')
    k = np.asarray(loo.pareto_k).ravel()
    threshold = float(getattr(loo, 'good_k', .7))
    rows = pd.DataFrame({'base_sample_id': design['data'].base_sample_id,
                         'pareto_k': k, 'threshold': threshold,
                         'flag_for_refit': (~np.isfinite(k)) | (k>threshold)})
    write_csv(folder/'PSIS_LOO_patients.csv', rows)
    write_json(folder/'PSIS_LOO_summary.json', {
        'elpd_loo': float(loo.elpd_loo), 'se': float(loo.se), 'p_loo': float(loo.p_loo),
        'warning': bool(loo.warning), 'flagged_patients': int(rows.flag_for_refit.sum()),
        'unit': 'Patient: both protein and PTM held out together',
        'prediction_target': 'Another patient in an observed plex; not a new plex'})
    return rows

def sample_model(model, config, seed, tune, draws):
    import pymc as pm
    if config['SAMPLER'] != 'pymc':
        raise ValueError("This notebook uses the tested PyMC NUTS sampler; set SAMPLER='pymc'.")
    with model:
        idata = pm.sample(draws=draws, tune=tune, chains=config['CHAINS'],
            cores=config['CORES'], random_seed=seed, target_accept=config['TARGET_ACCEPT'],
            nuts_sampler='pymc', nuts={'max_treedepth':config['MAX_TREEDEPTH']},
            mp_ctx='forkserver',                      # start clean workers instead of forking the kernel
            return_inferencedata=True, idata_kwargs={'log_likelihood':True},
            progressbar=True, compute_convergence_checks=True)
        if 'log_likelihood' not in idata.groups():
            idata = pm.compute_log_likelihood(idata, model=model, var_names=['paired_score'],
                extend_inferencedata=True, progressbar=False)
    return idata, 'pymc', None

In [ ]:
def load_saved_trace(path,chains,draws):
    obj=az.from_netcdf(path)
    try:
        obj.load()
        if obj.posterior.sizes['chain']!=chains or obj.posterior.sizes['draw']!=draws:
            raise ValueError('Saved trace dimensions do not match planned sampling: '+str(path))
    finally:obj.close()
    return obj

def sample_checkpointed(model,config,seed,tune,draws,folder,attempt):
    # Checkpoint completed chain batches; partial NUTS trajectories are not resumed.
    checkpoint=folder/f'attempt_{attempt}_chains';checkpoint.mkdir(parents=True,exist_ok=True)
    plan_file=checkpoint/'chain_plan.json'
    identity=dict(seed=seed,tune=tune,draws=draws,chains=config['CHAINS'])
    if plan_file.exists():
        plan=json.loads(plan_file.read_text())
        if plan['identity']!=identity:raise ValueError('Chain checkpoint settings mismatch')
    else:
        size=max(1,min(config['CORES'],config['CHAINS']))
        batches=[list(range(i,min(i+size,config['CHAINS']))) for i in range(0,config['CHAINS'],size)]
        seeds=[int(s.generate_state(1)[0])%(2**31-1) for s in np.random.SeedSequence(seed).spawn(len(batches))]
        plan={'identity':identity,'batches':batches,'seeds':seeds};write_json(plan_file,plan)
    pieces=[]
    for i,(chains,batch_seed) in enumerate(zip(plan['batches'],plan['seeds'])):
        dest=checkpoint/f'batch_{i:02d}.nc'
        if dest.exists():
            print('RESUME saved chains',chains,flush=True)
            part=load_saved_trace(dest,len(chains),draws)
        else:
            write_json(folder/'progress.json',dict(stage='sampling',attempt=attempt,chains=chains,time=time.time()))
            local=dict(config,CHAINS=len(chains),CORES=min(config['CORES'],len(chains)))
            print('Sampling and saving chains',chains,flush=True)
            part,_,_=sample_model(model,local,batch_seed,tune,draws)
            tmp=dest.with_name(dest.name+'.partial.nc');part.to_netcdf(tmp);os.replace(tmp,dest)
        pieces.append(part)
    combined=az.concat(*pieces,dim='chain',reset_dim=True) if len(pieces)>1 else pieces[0]
    if combined.posterior.sizes['chain']!=config['CHAINS']:raise ValueError('Missing saved chains')
    return combined

def checkpoint_complete(folder):
    status=folder/'status.json'
    if not status.exists():return False
    try:info=json.loads(status.read_text())
    except (ValueError,OSError):return False
    required=['effects_95HDI.csv','predictive_checks.csv']
    if folder.name=='primary':required+=['PSIS_LOO_patients.csv','PSIS_LOO_summary.json']
    return bool(info.get('postprocessing_complete') and all((folder/f).is_file() for f in required)
        and (folder/info.get('posterior_file','__missing__')).is_file())

def save_job_progress(run):
    rows=[]
    for job in JOBS[JOBS.shard.eq(SHARD_ID)].to_dict('records'):
        folder=run/'fits'/job['key']/job['variant']
        if checkpoint_complete(folder):state='complete'
        elif list(folder.glob('posterior_attempt_*.nc')):state='posterior_saved_postprocessing_pending'
        elif list(folder.glob('attempt_*_chains/batch_*.nc')):state='some_chains_saved'
        elif (folder/'progress.json').exists():state='interrupted_or_running'
        else:state='pending'
        rows.append(dict(job,state=state))
    write_csv(run/'job_progress.csv',rows)
    return pd.DataFrame(rows)

def fit_one(feature,variant,run,leave_out=None):
    key=feature['key'];d=feature_cohort(DATA,feature['assay']);a=ARRAYS[key]
    mask,adj,extra,cat=variant_spec(d,a['mask'],variant)
    if leave_out is not None:mask &= d.base_sample_id.ne(leave_out).to_numpy()
    design=make_design(d,mask,adj,extra,cat);observed=a['Y'][mask]
    label=variant if leave_out is None else variant+'__without_'+leave_out
    folder=run/'fits'/key/label;folder.mkdir(parents=True,exist_ok=True)
    status=folder/'status.json'
    if checkpoint_complete(folder):
        print('SKIP completed',key,label,flush=True)
        return json.loads(status.read_text())
    seed=int(hashlib.sha256((key+label+str(CONFIG['SEED'])).encode()).hexdigest()[:8],16)%(2**31-1)
    model=build_model(design,observed,CONFIG)
    if not np.isfinite(float(model.compile_logp()(model.initial_point()))):raise ValueError('Non-finite initial density')
    write_csv(folder/'design_matrix.csv',pd.DataFrame(design['X'],columns=design['names']).assign(base_sample_id=design['data'].base_sample_id))
    write_json(folder/'design_notes.json',{'dropped_constant_terms':design['dropped_constant_terms'],'scalers_frozen_from_primary':True})
    start=time.monotonic()
    for attempt in range(2 if CONFIG['RETRY_ONCE'] else 1):
        trace=folder/f'posterior_attempt_{attempt+1}.nc'
        if trace.exists():
            print('RESUME saved posterior',trace.name,flush=True)
            idata=load_saved_trace(trace,CONFIG['CHAINS'],CONFIG['DRAWS']*2**attempt)
        else:
            print('Fit',key,label,'n=',len(observed),'attempt',attempt+1)
            idata=sample_checkpointed(model,CONFIG,seed+attempt*1009,CONFIG['TUNE']*2**attempt,CONFIG['DRAWS']*2**attempt,folder,attempt+1)
            temp=trace.with_name(trace.name+'.partial.nc');idata.to_netcdf(temp);os.replace(temp,trace)
        diagnostics=numerical_diagnostics(idata,model,CONFIG,folder)
        if diagnostics['numerical_checks_pass'] or PROFILE=='smoke':break
    write_csv(folder/'effects_95HDI.csv',effect_summary(idata,feature,label,design,diagnostics['numerical_checks_pass']))
    info=dict(key=key,assay=feature['assay'],variant=label,postprocessing_complete=False,
        posterior_file=trace.name,attempt=attempt+1,seconds_this_session=time.monotonic()-start,**diagnostics)
    write_json(status,info)
    info['predictive_flags']=compute_ppc(idata,design,observed,CONFIG,folder,seed)
    info['PSIS_flagged_patients']=0
    if variant=='primary' and leave_out is None:
        psis=compute_loo(idata,design,folder);info['PSIS_flagged_patients']=int(psis.flag_for_refit.sum())
    info['postprocessing_complete']=True;write_json(status,info)
    write_json(folder/'progress.json',{'stage':'complete','time':time.time()})
    del idata
    return info

def run_deletions(feature,run):
    source=run/'fits'/feature['key']/'primary'/'PSIS_LOO_patients.csv'
    if not source.exists():return
    psis=pd.read_csv(source)
    if RUN_INFLUENCE_REFITS and PROFILE!='smoke':
        for row in psis[truth(psis.flag_for_refit)].itertuples():
            try:fit_one(feature,'primary',run,str(row.base_sample_id))
            except ValueError as exc:
                write_json(run/'fits'/feature['key']/('primary__without_'+str(row.base_sample_id))/'deletion_error.json',{'error':str(exc)})


In [ ]:
def fingerprint_analysis(function_names):
    # Ignore cell filenames and line numbers so re-executing a cell does not change its identity.
    def pack(value):
        if isinstance(value, types.CodeType):
            return dict(bytecode=value.co_code.hex(), constants=[pack(x) for x in value.co_consts],
                        names=list(value.co_names), variables=list(value.co_varnames),
                        freevars=list(value.co_freevars), cellvars=list(value.co_cellvars),
                        arguments=value.co_argcount, positional=value.co_posonlyargcount,
                        keyword_only=value.co_kwonlyargcount, flags=value.co_flags)
        if isinstance(value, (set, frozenset)):
            return {'unordered':sorted([pack(x) for x in value],key=lambda x:json.dumps(x,sort_keys=True))}
        if isinstance(value, bytes): return {'bytes': value.hex()}
        if isinstance(value, tuple): return [pack(x) for x in value]
        if value is None or isinstance(value, (str, int, float, bool)): return value
        return repr(value)
    return canonical_hash({name:pack(globals()[name].__code__) for name in function_names})
LEGACY_FUNCTION_SOURCE='def sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for part in iter(lambda: handle.read(2**20), b\'\'):\n            h.update(part)\n    return h.hexdigest()\n\ndef canonical_hash(obj):\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, allow_nan=False,\n                                    separators=(\',\', \':\')).encode()).hexdigest()\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + \'.partial\')\n    temp.write_text(json.dumps(obj, indent=2, allow_nan=False, default=str))\n    os.replace(temp, path)\n\ndef write_csv(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + \'.partial\')\n    pd.DataFrame(data).to_csv(temp, index=False)\n    os.replace(temp, path)\n\ndef truth(series):\n    return series.astype(str).str.strip().str.lower().isin([\'true\', \'1\', \'yes\'])\n\ndef unpack_inputs(path,work):\n    if not path.is_file(): raise FileNotFoundError(path)\n    digest=sha256_file(path); dest=work/(\'inputs_\'+digest[:16]);dest.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(path) as z:\n        names=z.namelist()\n        manifests=[s for s in names if s.endswith(\'/package_manifest.json\')]\n        if len(manifests)!=1: raise ValueError(\'Expected one package manifest\')\n        prefix=manifests[0].removesuffix(\'package_manifest.json\')\n        manifest=json.loads(z.read(manifests[0]))\n        for item in manifest:\n            rel=item[\'path\']; target=(dest/rel).resolve()\n            if not target.is_relative_to(dest.resolve()): raise ValueError(\'Unsafe archive path\')\n            content=z.read(prefix+rel)\n            if len(content)!=item[\'bytes\'] or hashlib.sha256(content).hexdigest()!=item[\'sha256\']:\n                raise ValueError(\'Input checksum failure: \'+rel)\n            target.parent.mkdir(parents=True,exist_ok=True)\n            if not target.exists() or sha256_file(target)!=item[\'sha256\']: target.write_bytes(content)\n    return dest,digest\n\ndef load_signature_links(path,input_hash):\n    if not path.is_file():\n        if MODE==\'preflight\':return pd.DataFrame(),None,{\'status\':\'missing original-signature ZIP; full fits blocked\'}\n        raise FileNotFoundError(f\'Use the same signature ZIP as your RNA–protein notebook: {path}\')\n    if sha256_file(path)!=EXPECTED_SIGNATURE_ZIP_SHA256: raise ValueError(\'Original signature ZIP changed\')\n    with zipfile.ZipFile(path) as z:\n        payload=json.loads(z.read(\'payload.json\'))\n        if payload[\'input_zip_sha256\']!=input_hash: raise ValueError(\'Cohort ZIP differs from archived original run\')\n        content=z.read(payload[\'signature_file\'])\n        if hashlib.sha256(content).hexdigest()!=payload[\'signature_sha256\']: raise ValueError(\'Signature checksum mismatch\')\n        signatures=json.loads(content)\n    inv=pd.DataFrame(payload[\'original_features\'])\n    if len(inv)!=914 or not inv.global_feature.is_unique: raise ValueError(\'Expected 914 original labels\')\n    rows=[]\n    for f in inv.to_dict(\'records\'):\n        raw=signatures[f[\'CellType\']][f[\'Signature\']]\n        if not isinstance(raw,list) or not all(isinstance(g,str) for g in raw): raise ValueError(\'Unsigned original schema changed\')\n        genes=sorted(set(g.strip().upper() for g in raw if g.strip()))\n        h=hashlib.sha256((\'POS:\'+\',\'.join(genes)+\'|NEG:\').encode()).hexdigest()\n        if h!=f[\'source_gene_hash\']: raise ValueError(\'Source gene list hash mismatch\')\n        for gene in genes: rows.append({k:f[k] for k in [\'global_feature\',\'family\',\'CellType\',\'Signature\',\'source_gene_hash\']}|{\'gene\':gene})\n    return pd.DataFrame(rows),payload,{\'status\':\'verified\',\'original_labels\':914}\n\ndef load_data(root,payload):\n    p=root/\'prepared\'; cohort=pd.read_csv(p/\'model_covariates.csv\')\n    ids=cohort.base_sample_id.astype(str).tolist()\n    ref=pd.read_csv(p/\'main_cohort_127.csv\')\n    assert len(ids)==len(set(ids))==127 and ids==ref.base_sample_id.tolist()\n    assert np.isfinite(cohort[[\'bmi\',\'age\',\'sex_centered\']]).all().all()\n    assert cohort.bmi_group.value_counts().to_dict()=={\'overweight\':57,\'normal_weight\':52,\'obese\':18}\n    contract=json.loads((p/\'preparation_contract.json\').read_text())\n    assert sha256_file(root/\'provenance/analysis_cohort.csv\')==contract[\'main_cohort_sha256\']\n    for col in [\'adenosquamous_recorded\',\'weight_loss_recorded\']:cohort[col]=truth(cohort[col])\n    if payload is not None:\n        purity=pd.DataFrame(payload[\'dna_purity_records\'])\n        cohort=cohort.merge(purity,on=\'base_sample_id\',how=\'left\',validate=\'one_to_one\',sort=False)\n        for col in [\'wes_purity\',\'wgs_purity\']:\n            assert cohort[col].dropna().between(0,1).all()\n    mutation=pd.read_csv(p/\'matrices/mutation_tumor_main_cohort.tsv.gz\',sep=\'\\t\',index_col=0,keep_default_na=False)\n    nonsyn={\'missense_variant\',\'frameshift_variant\',\'stop_gained\',\'stop_lost\',\'start_lost\',\n        \'inframe_deletion\',\'inframe_insertion\',\'splice_donor_variant\',\'splice_acceptor_variant\'}\n    import re\n    for gene in [\'TP53\',\'CDKN2A\',\'SMAD4\']:\n        calls=mutation.loc[gene,ids]\n        if calls.eq(\'\').any(): raise ValueError(\'Missing mutation status\')\n        cohort[\'mutation_\'+gene]=[int(bool(set(re.split(\'[,;]\',s))&nonsyn)) for s in calls]\n    plex=pd.read_csv(p/\'tumor_plex_map.csv\')\n    assert not plex.duplicated([\'layer\',\'base_sample_id\']).any()\n    by_layer={}\n    for layer in [\'protein\']+ASSAYS:\n        s=plex[plex.layer.eq(layer)].set_index(\'base_sample_id\').loc[ids]\n        assert len(s)==127 and s.plex.notna().all()\n        by_layer[layer]=s.plex.to_numpy()\n    # Current source has identical patient batch partitions across assays, even when names differ.\n    for layer in ASSAYS:\n        a=by_layer[layer]; b=by_layer[\'protein\']\n        if not np.array_equal(a[:,None]==a[None,:],b[:,None]==b[None,:]):\n            raise ValueError(\'Assay batch partitions differ: needs an explicitly separate-batch model\')\n    matrices={}; annotations={}\n    for layer in [\'protein\']+ASSAYS:\n        with np.load(p/f\'matrices/{layer}_tumor_main_cohort.npz\',allow_pickle=False) as z:\n            assert z[\'patient_ids\'].tolist()==ids\n            values=z[\'values\'].astype(float);features=z[\'feature_ids\'].astype(str)\n        assert values.shape==(len(features),127) and len(set(features))==len(features)\n        assert not np.isinf(values).any()\n        ann=pd.read_csv(p/f\'feature_metadata/{layer}_annotations.csv\',keep_default_na=False)\n        assert ann.feature_id.tolist()==features.tolist()\n        matrices[layer]={\'values\':values,\'feature_ids\':features}\n        annotations[layer]=ann\n    return dict(cohort=cohort,matrices=matrices,annotations=annotations,batches=by_layer,contract=contract)\n\ndef make_design(cohort,mask,adjustment=\'mundlak\',extra=None,categorical=False):\n    d=cohort.loc[mask].copy().reset_index(drop=True)\n    levels=sorted(d.plex.unique());group=pd.Categorical(d.plex,categories=levels).codes\n    raw=({\'overweight\':d.bmi_group.eq(\'overweight\').to_numpy(float),\'obese\':d.bmi_group.eq(\'obese\').to_numpy(float)}\n         if categorical else {\'bmi_5\':d.bmi.to_numpy()/5})\n    focal=list(raw)\n    raw.update(age_10y=d.age.to_numpy()/10,sex=d.sex_centered.to_numpy())\n    if extra==\'stage\':\n        for stage in [\'II\',\'III\',\'IV\']:raw[\'stage_\'+stage]=d.stage_major.eq(stage).to_numpy(float)\n    elif extra==\'mutations\':\n        for gene in [\'TP53\',\'CDKN2A\',\'SMAD4\']:raw[\'mutation_\'+gene]=d[\'mutation_\'+gene].to_numpy(float)\n    elif extra==\'wes_purity\':raw[\'wes_purity_10pct\']=d.wes_purity.to_numpy()/0.1\n    names=[];columns=[];between=[];dropped=[]\n    for key,x in raw.items():\n        if not np.isfinite(x).all():raise ValueError(\'Non-finite covariate \'+key)\n        mean=pd.Series(x).groupby(group).transform(\'mean\').to_numpy();v=x-mean\n        if v.std()<1e-10:\n            if key in focal:raise ValueError(\'No within-plex variation for \'+key)\n            dropped.append(key+\'_within\')\n        else:names.append(key+\'_within\');columns.append(v)\n        if adjustment==\'mundlak\':between.append((key+\'_between\',mean-x.mean()))\n    for name,value in between:\n        if value.std()<1e-10:dropped.append(name)\n        else:names.append(name);columns.append(value)\n    X=np.column_stack(columns)\n    Q=helmert(len(levels),full=False).T if adjustment==\'fixed\' else None\n    full=np.column_stack([np.ones(len(d)),X]+([Q[group]] if Q is not None else []))\n    if np.linalg.matrix_rank(full)!=full.shape[1]:raise ValueError(\'Rank-deficient design; no confounder deletion\')\n    return dict(data=d,X=X,names=names,group=group,levels=levels,Q=Q,adjustment=adjustment,extra=extra,\n        dropped_constant_terms=dropped,focal=[s+\'_within\' for s in focal],categorical=categorical)\n\ndef prepare_features(data,links,qc_path=None):\n    protein=data[\'matrices\'][\'protein\'];lookup={g:i for i,g in enumerate(protein[\'feature_ids\'])}\n    genes=set(links.gene) if len(links) else set();audits=[];records=[];arrays={}\n    qc=None\n    if qc_path:\n        qc=pd.read_csv(qc_path,keep_default_na=False)\n        if not {\'assay\',\'feature_id\',\'pass_qc\',\'provenance\'}<=set(qc):raise ValueError(\'Invalid QC schema\')\n        if qc.duplicated([\'assay\',\'feature_id\']).any():raise ValueError(\'Duplicate QC feature\')\n        if qc.provenance.eq(\'\').any():raise ValueError(\'QC provenance missing\')\n        qc={(r.assay,r.feature_id):str(r.pass_qc).lower() in (\'true\',\'1\',\'yes\') for r in qc.itertuples()}\n    for assay in ASSAYS:\n        matrix=data[\'matrices\'][assay];ann=data[\'annotations\'][assay]\n        for row in ann.itertuples(index=True):\n            sid=row.feature_id;gene=row.Gene.strip().upper();reason=\'\';n=0;nplex=0\n            target=(sid in SENSITIVITY_TARGETS[\'phospho\'] if assay==\'phospho\' else gene in SENSITIVITY_TARGETS[\'glyco_genes\'])\n            if FEATURE_SCOPE==\'original_signatures_plus_targets\' and gene not in genes and not target:\n                reason=\'outside original genes and named targets\' if len(links) else \'signature archive unavailable; scope unresolved\'\n            elif qc is not None and not qc.get((assay,sid),False):reason=\'external QC not passed or absent\'\n            elif gene not in lookup:reason=\'parent protein unavailable by exact gene symbol\'\n            else:\n                pair=np.column_stack([protein[\'values\'][lookup[gene]],matrix[\'values\'][row.Index]])\n                mask=np.isfinite(pair).all(1);n=int(mask.sum());nplex=len(set(data[\'batches\'][assay][mask]))\n                if n<MIN_PAIRED_N or nplex<MIN_PLEXES:reason=\'paired coverage below threshold\'\n                elif np.any(pair[mask].std(0,ddof=1)<1e-8):reason=\'constant paired response\'\n            rec=dict(assay=assay,feature_id=sid,gene=gene,n_paired=n,n_plexes=nplex,target=target,\n                included=not bool(reason),exclusion_reason=reason,localization_policy=\'external_QC\' if qc is not None else \'source_processed_unverified_localization\')\n            audits.append(rec)\n            if reason:continue\n            mean=pair[mask].mean(0);sd=pair[mask].std(0,ddof=1);Y=(pair-mean)/sd\n            key=assay+\'_\'+hashlib.sha256(sid.encode()).hexdigest()[:20]\n            arrays[key]={\'Y\':Y,\'mask\':mask,\'means\':mean,\'sds\':sd}\n            records.append(rec|dict(key=key,protein_row=gene,protein_mean=float(mean[0]),protein_sd=float(sd[0]),\n                ptm_mean=float(mean[1]),ptm_sd=float(sd[1]),parent_resolution=\'gene_level\'))\n    return pd.DataFrame(records),pd.DataFrame(audits),arrays\n\ndef feature_cohort(data,assay):\n    d=data[\'cohort\'].copy();d[\'plex\']=data[\'batches\'][assay];return d\n\ndef variant_spec(d,base,variant):\n    mask=base.copy();extra=None;adjustment=\'mundlak\';categorical=False\n    if variant==\'fixed_plex\':adjustment=\'fixed\'\n    elif variant in (\'stage_subset_base\',\'stage_adjusted\'):\n        mask &= d.stage_major.isin([\'I\',\'II\',\'III\',\'IV\']).to_numpy()\n        if variant==\'stage_adjusted\':extra=\'stage\'\n    elif variant in (\'wes_subset_base\',\'wes_adjusted\'):\n        if \'wes_purity\' not in d:raise ValueError(\'WES purity absent from original payload\')\n        mask &= np.isfinite(d.wes_purity.to_numpy())\n        if variant==\'wes_adjusted\':extra=\'wes_purity\'\n    elif variant==\'mutation_adjusted\':extra=\'mutations\'\n    elif variant==\'exclude_recorded_weight_loss\':mask &= ~d.weight_loss_recorded.to_numpy(bool)\n    elif variant==\'exclude_adenosquamous\':mask &= ~d.adenosquamous_recorded.to_numpy(bool)\n    elif variant==\'categorical\':categorical=True\n    elif variant!=\'primary\':raise ValueError(variant)\n    if mask.sum()<30:raise ValueError(\'Fewer than 30 patients in sensitivity subset\')\n    return mask,adjustment,extra,categorical\n\ndef plan_jobs(data,features,arrays):\n    jobs=[];excluded=[]\n    variants=[\'fixed_plex\',\'stage_subset_base\',\'stage_adjusted\',\'wes_subset_base\',\'wes_adjusted\',\n        \'mutation_adjusted\',\'exclude_recorded_weight_loss\',\'exclude_adenosquamous\']\n    if RUN_CATEGORICAL:variants+=[\'categorical\']\n    for f in features.to_dict(\'records\'):\n        d=feature_cohort(data,f[\'assay\'])\n        for variant in [\'primary\']+(variants if f[\'target\'] and RUN_SENSITIVITIES and PROFILE!=\'smoke\' else []):\n            try:\n                mask,adj,extra,cat=variant_spec(d,arrays[f[\'key\']][\'mask\'],variant)\n                des=make_design(d,mask,adj,extra,cat)\n                jobs.append(dict(key=f[\'key\'],assay=f[\'assay\'],variant=variant,n_patients=int(mask.sum()),\n                    n_plexes=len(des[\'levels\']),shard=int(hashlib.sha256(f[\'key\'].encode()).hexdigest()[:12],16)%N_SHARDS))\n            except ValueError as exc:\n                excluded.append(dict(key=f[\'key\'],variant=variant,reason=str(exc)))\n                if variant==\'primary\':break\n    return pd.DataFrame(jobs),pd.DataFrame(excluded)\n\ndef build_model(design, observed, config):\n    import pymc as pm\n    import pytensor.tensor as pt\n    coords = {\'patient\': design[\'data\'].base_sample_id.tolist(), \'layer\': LAYERS,\n              \'term\': design[\'names\'], \'plex\': design[\'levels\']}\n    if design[\'adjustment\'] == \'fixed\':\n        coords[\'plex_contrast\'] = list(range(len(design[\'levels\'])-1))\n    prior = np.full(len(design[\'names\']), config[\'COVARIATE_PRIOR_SD\'])\n    for i, name in enumerate(design[\'names\']):\n        if name.startswith(\'bmi_5_\'): prior[i] = config[\'BMI_PRIOR_SD\']\n    with pm.Model(coords=coords) as model:\n        alpha = pm.Normal(\'alpha\', 0, 0.5, dims=\'layer\')\n        coefficient = pm.Normal(\'coefficient\', 0, prior[:, None], dims=(\'term\', \'layer\'))\n        if design[\'adjustment\'] == \'mundlak\':\n            group_scale = pm.HalfNormal(\'group_scale\', config[\'GROUP_SD_PRIOR\'], dims=\'layer\')\n            group_z = pm.Normal(\'group_z\', 0, 1, dims=(\'plex\', \'layer\'))\n            group_effect = pm.Deterministic(\'group_effect\', group_z*group_scale,\n                                           dims=(\'plex\', \'layer\'))\n        else:\n            group_coefficient = pm.Normal(\'group_coefficient\', 0,\n                                           config[\'FIXED_GROUP_PRIOR_SD\'],\n                                           dims=(\'plex_contrast\', \'layer\'))\n            group_effect = pm.Deterministic(\'group_effect\',\n                pt.dot(design[\'Q\'], group_coefficient), dims=(\'plex\', \'layer\'))\n        mu = alpha + pt.dot(design[\'X\'], coefficient) + group_effect[design[\'group\']]\n        chol, corr, scales = pm.LKJCholeskyCov(\'resid_chol\', n=2, eta=2,\n            sd_dist=pm.HalfNormal.dist(1.0, shape=2), compute_corr=True)\n        nu_minus_two = pm.Exponential(\'nu_minus_two\', 0.1)\n        nu = pm.Deterministic(\'nu\', 2+nu_minus_two)\n        positions=[design[\'names\'].index(s) for s in design[\'focal\']]\n        slopes=coefficient[positions]\n        coupling=pm.Deterministic(\'protein_to_ptm_coupling\',corr[0,1]*scales[1]/scales[0])\n        pm.Deterministic(\'focal_effect\',slopes)\n        pm.Deterministic(\'ptm_minus_protein\',slopes[:,1]-slopes[:,0])\n        pm.Deterministic(\'ptm_conditional_effect\',slopes[:,1]-coupling*slopes[:,0])\n        pm.Deterministic(\'residual_correlation\',corr[0,1])\n        pm.MvStudentT(\'paired_score\', nu=nu, mu=mu, chol=chol,\n                      observed=observed, dims=(\'patient\', \'layer\'))\n    return model\n\ndef posterior_array(idata, name):\n    da = idata.posterior[name].stack(sample=(\'chain\', \'draw\'))\n    return da.transpose(\'sample\', *[d for d in da.dims if d != \'sample\']).values\n\ndef hdi(values, probability=0.95):\n    a = np.sort(np.asarray(values, float).ravel())\n    if not np.isfinite(a).all() or len(a) < 20:\n        raise ValueError(\'Cannot summarize incomplete posterior draws\')\n    width = int(np.floor(probability*len(a)))\n    left = int(np.argmin(a[width:]-a[:-width]))\n    return float(a[left]), float(a[left+width])\n\ndef numerical_diagnostics(idata, model, config, folder):\n    import arviz as az\n    names = [v.name for v in model.free_RVs] + [\'focal_effect\',\'ptm_conditional_effect\',\'protein_to_ptm_coupling\']\n    summary = az.summary(idata, var_names=names, kind=\'diagnostics\', round_to=\'none\')\n    summary.rename_axis(\'parameter\').reset_index().to_csv(folder/\'parameter_diagnostics.csv\', index=False)\n    columns = summary[[\'r_hat\', \'ess_bulk\', \'ess_tail\']].to_numpy(float)\n    finite = bool(np.isfinite(columns).all())\n    rhat = float(np.nanmax(summary.r_hat))\n    bulk, tail = float(np.nanmin(summary.ess_bulk)), float(np.nanmin(summary.ess_tail))\n    stats = idata.sample_stats\n    divergences = int(stats[\'diverging\'].sum())\n    bfmi = float(np.nanmin(az.bfmi(idata)))\n    if \'reached_max_treedepth\' in stats:\n        depth_hits = int(stats[\'reached_max_treedepth\'].sum())\n    elif \'tree_depth\' in stats:\n        depth_hits = int((stats.tree_depth >= config[\'MAX_TREEDEPTH\']).sum())\n    elif \'depth\' in stats:\n        depth_hits = int((stats.depth >= config[\'MAX_TREEDEPTH\']).sum())\n    else:\n        raise RuntimeError(\'Sampler returned no tree-depth statistic; cannot verify the gate\')\n    passed = finite and rhat <= 1.01 and min(bulk, tail) >= 400 and bfmi >= 0.30 \\\n             and divergences == 0 and depth_hits == 0\n    return {\'max_rhat\': rhat if np.isfinite(rhat) else None,\n            \'min_ess_bulk\': bulk if np.isfinite(bulk) else None,\n            \'min_ess_tail\': tail if np.isfinite(tail) else None,\n            \'min_bfmi\': bfmi if np.isfinite(bfmi) else None,\n            \'divergences\': divergences, \'maximum_depth_hits\': depth_hits,\n            \'all_diagnostics_finite\': finite, \'numerical_checks_pass\': bool(passed),\n            \'publication_profile\': config[\'PROFILE\'] == \'publication\'}\n\ndef effect_summary(idata,feature,variant,design,passed):\n    beta=posterior_array(idata,\'focal_effect\');conditional=posterior_array(idata,\'ptm_conditional_effect\')\n    rows=[]\n    def add(quantity,draws,term,units):\n        lo,hi=hdi(draws)\n        rows.append(dict(key=feature[\'key\'],assay=feature[\'assay\'],feature_id=feature[\'feature_id\'],gene=feature[\'gene\'],\n            variant=variant,term=term,quantity=quantity,posterior_mean=float(np.mean(draws)),\n            hdi_95_lower=lo,hdi_95_upper=hi,hdi_excludes_zero=bool(lo>0 or hi<0),\n            probability_positive=float(np.mean(draws>0)),n_patients=len(design[\'data\']),n_plexes=len(design[\'levels\']),\n            numerical_checks_pass=bool(passed),effect_units=units,interval_type=\'pointwise 95% HDI; no multiplicity control\'))\n    for j,term in enumerate(design[\'focal\']):\n        units=\'primary-paired-sample response SD\'+(\' vs normal weight, within plex\' if design[\'categorical\'] else \' per 5 kg/m2, within plex\')\n        for label,values in [(\'Protein\',beta[:,j,0]),(\'PTM\',beta[:,j,1]),\n            (\'PTM_minus_Protein_standardized\',beta[:,j,1]-beta[:,j,0]),(\'PTM_given_Protein\',conditional[:,j])]:\n            add(label,values,term,units)\n        if COMPATIBLE_LOG2_RATIO:\n            add(\'log2_PTM_minus_Protein\',feature[\'ptm_sd\']*beta[:,j,1]-feature[\'protein_sd\']*beta[:,j,0],term,\n                \'log2 ratio\'+(\' vs normal weight\' if design[\'categorical\'] else \' per 5 kg/m2\'))\n    add(\'residual_Protein_PTM_correlation\',posterior_array(idata,\'residual_correlation\'),\'residual\',\'correlation\')\n    add(\'protein_to_ptm_coupling\',posterior_array(idata,\'protein_to_ptm_coupling\'),\'residual\',\'PTM SD per protein SD\')\n    return pd.DataFrame(rows)\n\ndef compute_ppc(idata, design, observed, config, folder, seed):\n    rng = np.random.default_rng(seed+231)\n    a, c, g = [posterior_array(idata, n) for n in [\'alpha\', \'coefficient\', \'group_effect\']]\n    total = len(a)\n    chosen = np.sort(rng.choice(total, min(config[\'PPC_DRAWS\'], total), replace=False))\n    mu = a[chosen,None,:] + np.einsum(\'nt,stl->snl\', design[\'X\'], c[chosen]) \\\n         + g[chosen][:, design[\'group\'], :]\n    scales = posterior_array(idata, \'resid_chol_stds\')[chosen]\n    corr = posterior_array(idata, \'resid_chol_corr\')[chosen]\n    nu = posterior_array(idata, \'nu\')[chosen]\n    S = corr*scales[:,:,None]*scales[:,None,:]\n    L = np.linalg.cholesky(S)\n    z = rng.normal(size=mu.shape)\n    gaussian = np.einsum(\'sij,snj->sni\', L, z)\n    w = np.sqrt(rng.chisquare(nu[:,None], size=mu.shape[:2])/nu[:,None])\n    replicate = mu + gaussian/w[:,:,None]\n    observed_residual = observed[None,:,:]-mu\n    replicate_residual = replicate-mu\n    checks = []\n    def add(name, obs, rep):\n        upper = float(np.mean(rep >= obs))\n        checks.append({\'check\': name, \'observed_draw_median\': float(np.median(obs)),\n                       \'replicated_median\': float(np.median(rep)),\n                       \'replicated_95_lower\': float(np.quantile(rep,.025)),\n                       \'replicated_95_upper\': float(np.quantile(rep,.975)),\n                       \'posterior_predictive_upper_tail_probability\': upper,\n                       \'flag\': bool(upper < .025 or upper > .975),\n                       \'interpretation\': \'Descriptive predictive check; not a calibrated p-value\'})\n    for j, layer in enumerate(LAYERS):\n        add(layer+\'_residual_SD\', observed_residual[:,:,j].std(1,ddof=1),\n            replicate_residual[:,:,j].std(1,ddof=1))\n        add(layer+\'_maximum_absolute_residual\', np.abs(observed_residual[:,:,j]).max(1),\n            np.abs(replicate_residual[:,:,j]).max(1))\n    def corrs(x):\n        v=x-x.mean(1,keepdims=True)\n        return np.sum(v[:,:,0]*v[:,:,1],axis=1)/np.sqrt(\n            np.sum(v[:,:,0]**2,axis=1)*np.sum(v[:,:,1]**2,axis=1))\n    add(\'Protein_PTM_residual_correlation\', corrs(observed_residual), corrs(replicate_residual))\n    low, high = np.quantile(replicate,[.05,.95],axis=0)\n    for j, layer in enumerate(LAYERS):\n        checks.append({\'check\': layer+\'_90pct_in_sample_predictive_coverage\',\n                       \'observed_draw_median\': float(np.mean((observed[:,j]>=low[:,j]) &\n                                                            (observed[:,j]<=high[:,j]))),\n                       \'interpretation\': \'In-sample descriptive coverage; not out-of-sample validation\',\n                       \'flag\': False})\n    write_csv(folder/\'predictive_checks.csv\', checks)\n    write_csv(folder/\'patient_residuals.csv\', pd.DataFrame({\n        \'base_sample_id\': design[\'data\'].base_sample_id,\n        \'Protein_residual\': observed[:,0]-mu[:,:,0].mean(0),\n        \'PTM_residual\': observed[:,1]-mu[:,:,1].mean(0)}))\n    return int(sum(bool(x.get(\'flag\',False)) for x in checks))\n\ndef compute_loo(idata, design, folder):\n    import arviz as az\n    ll = idata.log_likelihood[\'paired_score\']\n    point_dims = [d for d in ll.dims if d not in [\'chain\', \'draw\']]\n    if len(point_dims)!=1 or ll.sizes[point_dims[0]]!=len(design[\'data\']):\n        raise RuntimeError(\'LOO must have one joint protein/PTM likelihood per patient.\')\n    loo = az.loo(idata, pointwise=True, var_name=\'paired_score\')\n    k = np.asarray(loo.pareto_k).ravel()\n    threshold = float(getattr(loo, \'good_k\', .7))\n    rows = pd.DataFrame({\'base_sample_id\': design[\'data\'].base_sample_id,\n                         \'pareto_k\': k, \'threshold\': threshold,\n                         \'flag_for_refit\': (~np.isfinite(k)) | (k>threshold)})\n    write_csv(folder/\'PSIS_LOO_patients.csv\', rows)\n    write_json(folder/\'PSIS_LOO_summary.json\', {\n        \'elpd_loo\': float(loo.elpd_loo), \'se\': float(loo.se), \'p_loo\': float(loo.p_loo),\n        \'warning\': bool(loo.warning), \'flagged_patients\': int(rows.flag_for_refit.sum()),\n        \'unit\': \'Patient: both protein and PTM held out together\',\n        \'prediction_target\': \'Another patient in an observed plex; not a new plex\'})\n    return rows\n\ndef sample_model(model, config, seed, tune, draws):\n    import pymc as pm\n    if config[\'SAMPLER\'] != \'pymc\':\n        raise ValueError("This notebook uses the tested PyMC NUTS sampler; set SAMPLER=\'pymc\'.")\n    with model:\n        idata = pm.sample(draws=draws, tune=tune, chains=config[\'CHAINS\'],\n            cores=config[\'CORES\'], random_seed=seed, target_accept=config[\'TARGET_ACCEPT\'],\n            nuts_sampler=\'pymc\', nuts={\'max_treedepth\':config[\'MAX_TREEDEPTH\']},\n            mp_ctx=\'forkserver\',                      # start clean workers instead of forking the kernel\n            return_inferencedata=True, idata_kwargs={\'log_likelihood\':True},\n            progressbar=True, compute_convergence_checks=True)\n        if \'log_likelihood\' not in idata.groups():\n            idata = pm.compute_log_likelihood(idata, model=model, var_names=[\'paired_score\'],\n                extend_inferencedata=True, progressbar=False)\n    return idata, \'pymc\', None\n\ndef fit_one(feature,variant,run,leave_out=None):\n    key=feature[\'key\'];d=feature_cohort(DATA,feature[\'assay\']);a=ARRAYS[key]\n    mask,adj,extra,cat=variant_spec(d,a[\'mask\'],variant)\n    if leave_out is not None:mask &= d.base_sample_id.ne(leave_out).to_numpy()\n    design=make_design(d,mask,adj,extra,cat);observed=a[\'Y\'][mask]\n    label=variant if leave_out is None else variant+\'__without_\'+leave_out\n    folder=run/\'fits\'/key/label;folder.mkdir(parents=True,exist_ok=True)\n    status=folder/\'status.json\'\n    if status.exists() and json.loads(status.read_text()).get(\'postprocessing_complete\'):\n        return json.loads(status.read_text())\n    seed=int(hashlib.sha256((key+label+str(CONFIG[\'SEED\'])).encode()).hexdigest()[:8],16)%(2**31-1)\n    model=build_model(design,observed,CONFIG)\n    if not np.isfinite(float(model.compile_logp()(model.initial_point()))):raise ValueError(\'Non-finite initial density\')\n    write_csv(folder/\'design_matrix.csv\',pd.DataFrame(design[\'X\'],columns=design[\'names\']).assign(base_sample_id=design[\'data\'].base_sample_id))\n    write_json(folder/\'design_notes.json\',{\'dropped_constant_terms\':design[\'dropped_constant_terms\'],\'scalers_frozen_from_primary\':True})\n    start=time.monotonic()\n    for attempt in range(2 if CONFIG[\'RETRY_ONCE\'] else 1):\n        trace=folder/f\'posterior_attempt_{attempt+1}.nc\'\n        if trace.exists():\n            idata=az.from_netcdf(trace);idata.load();idata.close()\n        else:\n            print(\'Fit\',key,label,\'n=\',len(observed),\'attempt\',attempt+1)\n            idata,_,_=sample_model(model,CONFIG,seed+attempt*1009,CONFIG[\'TUNE\']*2**attempt,CONFIG[\'DRAWS\']*2**attempt)\n            temp=trace.with_name(trace.name+\'.partial.nc\');idata.to_netcdf(temp);os.replace(temp,trace)\n        diagnostics=numerical_diagnostics(idata,model,CONFIG,folder)\n        if diagnostics[\'numerical_checks_pass\'] or PROFILE==\'smoke\':break\n    write_csv(folder/\'effects_95HDI.csv\',effect_summary(idata,feature,label,design,diagnostics[\'numerical_checks_pass\']))\n    info=dict(key=key,assay=feature[\'assay\'],variant=label,postprocessing_complete=False,\n        posterior_file=trace.name,attempt=attempt+1,seconds_this_session=time.monotonic()-start,**diagnostics)\n    write_json(status,info)\n    info[\'predictive_flags\']=compute_ppc(idata,design,observed,CONFIG,folder,seed)\n    info[\'PSIS_flagged_patients\']=0\n    if variant==\'primary\' and leave_out is None:\n        psis=compute_loo(idata,design,folder);info[\'PSIS_flagged_patients\']=int(psis.flag_for_refit.sum())\n    info[\'postprocessing_complete\']=True;write_json(status,info)\n    del idata\n    return info\n\ndef run_deletions(feature,run):\n    source=run/\'fits\'/feature[\'key\']/\'primary\'/\'PSIS_LOO_patients.csv\'\n    if not source.exists():return\n    psis=pd.read_csv(source)\n    if RUN_INFLUENCE_REFITS and PROFILE!=\'smoke\':\n        for row in psis[truth(psis.flag_for_refit)].itertuples():\n            try:fit_one(feature,\'primary\',run,str(row.base_sample_id))\n            except ValueError as exc:\n                write_json(run/\'fits\'/feature[\'key\']/(\'primary__without_\'+str(row.base_sample_id))/\'deletion_error.json\',{\'error\':str(exc)})\n\ndef fingerprint_analysis(function_names):\n    # Ignore cell filenames and line numbers so re-executing a cell does not change its identity.\n    def pack(value):\n        if isinstance(value, types.CodeType):\n            return dict(bytecode=value.co_code.hex(), constants=[pack(x) for x in value.co_consts],\n                        names=list(value.co_names), variables=list(value.co_varnames),\n                        freevars=list(value.co_freevars), cellvars=list(value.co_cellvars),\n                        arguments=value.co_argcount, positional=value.co_posonlyargcount,\n                        keyword_only=value.co_kwonlyargcount, flags=value.co_flags)\n        if isinstance(value, bytes): return {\'bytes\': value.hex()}\n        if isinstance(value, tuple): return [pack(x) for x in value]\n        if value is None or isinstance(value, (str, int, float, bool)): return value\n        return repr(value)\n    return canonical_hash({name:pack(globals()[name].__code__) for name in function_names})\n\ndef linear_benchmark(d,Y):\n    groups=pd.Categorical(d.plex).codes\n    cov=np.column_stack([d.age.to_numpy()/10,d.sex_centered.to_numpy()])\n    dummy=np.eye(groups.max()+1)[groups]\n    bmi=d.bmi.to_numpy()/5\n    def coefficient(y,extra=None):\n        nuisance=np.column_stack([dummy,cov]+([] if extra is None else [extra]))\n        resid=bmi-nuisance@np.linalg.lstsq(nuisance,bmi,rcond=None)[0]\n        den=resid@resid\n        return float(resid@y/den) if den>1e-8 else np.nan\n    return np.array([coefficient(Y[:,0]),coefficient(Y[:,1]),coefficient(Y[:,1],Y[:,0])])\n\ndef bootstrap_targets(run):\n    if PROFILE==\'smoke\':return\n    targets=FEATURES[FEATURES.target];rng=np.random.default_rng(CONFIG[\'SEED\']+999)\n    group=DATA[\'batches\'][\'protein\'];parts=[np.flatnonzero(group==g) for g in sorted(set(group))]\n    resamples=[np.concatenate([rng.choice(ix,len(ix),replace=True) for ix in parts]) for _ in range(BOOTSTRAP_REPLICATES)]\n    rows=[]\n    for f in targets.to_dict(\'records\'):\n        a=ARRAYS[f[\'key\']];d=feature_cohort(DATA,f[\'assay\']);samples=[]\n        for ix in resamples:\n            ix=ix[a[\'mask\'][ix]]\n            samples.append(linear_benchmark(d.iloc[ix].reset_index(drop=True),a[\'Y\'][ix]))\n        samples=np.asarray(samples);point=linear_benchmark(d.loc[a[\'mask\']].reset_index(drop=True),a[\'Y\'][a[\'mask\']])\n        for j,quantity in enumerate([\'Protein\',\'PTM\',\'PTM_given_Protein\']):\n            v=samples[:,j];v=v[np.isfinite(v)]\n            lo,hi=np.quantile(v,[.025,.975]) if len(v)>=100 else (np.nan,np.nan)\n            rows.append(dict(key=f[\'key\'],feature_id=f[\'feature_id\'],assay=f[\'assay\'],quantity=quantity,\n                estimate=point[j],ci_95_lower=lo,ci_95_upper=hi,valid_replicates=len(v),requested=BOOTSTRAP_REPLICATES,\n                interval_type=\'percentile 95% CI, fixed-plex linear benchmark\'))\n    write_csv(run/\'bootstrap_target_linear_benchmark.csv\',rows)\n\ndef aggregate_outputs(root,run,contract):\n    effects=[];status=[];ppcs=[];influence=[];issues=[]\n    for shard in range(N_SHARDS):\n        src=root/f\'shard_{shard:02d}\'\n        if not (src/\'shard_complete.json\').exists():raise RuntimeError(f\'Shard {shard} incomplete\')\n        if json.loads((src/\'contract.json\').read_text())!=contract:raise RuntimeError(\'Shard contract mismatch\')\n        expected=JOBS[JOBS.shard.eq(shard)]\n        marker=json.loads((src/\'shard_complete.json\').read_text())\n        if marker[\'jobs\']!=expected[[\'key\',\'variant\']].to_dict(\'records\'):raise RuntimeError(\'Shard ownership mismatch\')\n        for job in expected.to_dict(\'records\'):\n            folder=src/\'fits\'/job[\'key\']/job[\'variant\'];s=json.loads((folder/\'status.json\').read_text())\n            if not s.get(\'postprocessing_complete\'):raise RuntimeError(\'Incomplete fit \'+str(folder))\n            status.append(s);base=pd.read_csv(folder/\'effects_95HDI.csv\');effects.append(base)\n            ppcs.append(pd.read_csv(folder/\'predictive_checks.csv\').assign(key=job[\'key\'],variant=job[\'variant\']))\n            if job[\'variant\']==\'primary\':\n                psis=pd.read_csv(folder/\'PSIS_LOO_patients.csv\')\n                for pid in psis.loc[truth(psis.flag_for_refit),\'base_sample_id\']:\n                    deleted=folder.parent/(\'primary__without_\'+pid)\n                    state=deleted/\'status.json\'\n                    adequate=state.exists() and json.loads(state.read_text()).get(\'postprocessing_complete\',False) and json.loads(state.read_text()).get(\'numerical_checks_pass\',False)\n                    issues.append(dict(key=job[\'key\'],patient=pid,deletion_numerically_adequate=adequate))\n                    if adequate:\n                        dt=pd.read_csv(deleted/\'effects_95HDI.csv\')\n                        comparison=base.merge(dt,on=[\'key\',\'quantity\',\'term\'],suffixes=(\'_primary\',\'_deletion\'))\n                        comparison[\'deleted_patient\']=pid\n                        comparison[\'posterior_mean_change\']=comparison.posterior_mean_deletion-comparison.posterior_mean_primary\n                        influence.append(comparison)\n    result=pd.concat(effects,ignore_index=True)\n    if result.duplicated([\'key\',\'variant\',\'quantity\',\'term\']).any():raise ValueError(\'Duplicated result rows\')\n    write_csv(run/\'all_effects_95HDI.csv\',result);write_csv(run/\'fit_status.csv\',status)\n    write_csv(run/\'predictive_checks.csv\',pd.concat(ppcs,ignore_index=True))\n    write_csv(run/\'influence_status.csv\',issues)\n    if influence:write_csv(run/\'patient_deletion_changes.csv\',pd.concat(influence,ignore_index=True))\n    if len(LINKS):\n        # Expanded label table is annotation, not independent observations or a discovery count.\n        write_csv(run/\'primary_effects_linked_to_914_labels.csv\',result[result.variant.eq(\'primary\')].merge(LINKS,on=\'gene\',how=\'inner\'))\n    summary=dict(planned_fits=len(JOBS),finished_fits=len(status),numerical_pass=sum(s[\'numerical_checks_pass\'] for s in status),\n        predictive_flags=sum(s[\'predictive_flags\'] for s in status),psis_flagged_pairs=len(issues),\n        unresolved_deletion_checks=sum(not x[\'deletion_numerically_adequate\'] for x in issues),\n        reporting=\'Pointwise 95% HDIs only; no ROPE or panel-wide FDR control.\',\n        influence_note=\'Adequate deletion refits require effect-change review; they do not repair PSIS scores.\',\n        calibration=\'No PTM interval-calibration simulation completed by this notebook.\',\n        annotation_policy=\'Source processed, localization probabilities unavailable\' if SITE_QC_CSV is None else \'External QC table\',\n        scope=\'Bulk gene-level parent protein paired with individual PTM features; no causal or occupancy claim\')\n    write_json(run/\'completion_summary.json\',summary)\n    return result,summary\n\ndef plot_and_package(result,run):\n    import matplotlib.pyplot as plt\n    primary=result[(result.variant==\'primary\')&result.numerical_checks_pass]\n    fig,axs=plt.subplots(1,len(ASSAYS),figsize=(6*len(ASSAYS),5),squeeze=False)\n    for ax,assay in zip(axs[0],ASSAYS):\n        w=primary[primary.assay.eq(assay)].pivot(index=\'key\',columns=\'quantity\',values=\'posterior_mean\')\n        if {\'PTM\',\'PTM_given_Protein\'}<=set(w):ax.scatter(w.PTM,w.PTM_given_Protein,s=8,alpha=.4)\n        ax.axhline(0,color=\'grey\',lw=.6);ax.axvline(0,color=\'grey\',lw=.6)\n        ax.set(title=assay,xlabel=\'Marginal PTM BMI slope\',ylabel=\'Protein-conditioned PTM BMI slope\')\n    fig.suptitle(\'Standardized response per 5 kg/m²; point estimates, no discovery classification\')\n    fig.tight_layout();fig.savefig(run/\'PTM_conditional_comparison.png\',dpi=180);plt.show()\n    targetkeys=set(FEATURES.loc[FEATURES.target,\'key\'])\n    display(primary[primary.key.isin(targetkeys)&primary.quantity.eq(\'PTM_given_Protein\')][\n        [\'assay\',\'feature_id\',\'posterior_mean\',\'hdi_95_lower\',\'hdi_95_upper\']])\n    dest=run/\'Protein_PTM_reports.zip\';tmp=dest.with_suffix(\'.partial.zip\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(run.iterdir()):\n            if p.is_file() and p not in (dest,tmp):z.write(p,p.name)\n    os.replace(tmp,dest);return dest'
def choose_resume_root(expected,legacy_hash):
    if RESUME_RUN_ROOT is not None:
        roots=[Path(RESUME_RUN_ROOT)]
    else:
        roots=sorted((OUTPUT_BASE/PROFILE).glob(f'*/shards_{N_SHARDS}'))
    matches=[];rejected=[]
    for root in roots:
        paths=sorted(root.glob('*/contract.json'))
        if not paths:continue
        contracts=[json.loads(p.read_text()) for p in paths]
        saved=contracts[0]
        if any(c!=saved for c in contracts):raise ValueError('Inconsistent saved contracts in '+str(root))
        differences=[k for k in expected if k!='code_hash' and expected[k]!=saved.get(k)]
        legacy_from_uploaded_notebook=(root.parent.name=='2687095c8b4e1e44' and canonical_hash(saved)[:16]=='2687095c8b4e1e44')
        if saved.get('code_hash') not in [legacy_hash,expected['code_hash']] and not legacy_from_uploaded_notebook:differences.append('code_hash')
        if differences:rejected.append((str(root),differences))
        else:matches.append((root,saved))
    if len(matches)>1:raise ValueError('Multiple compatible runs: set RESUME_RUN_ROOT explicitly: '+str([str(x[0]) for x in matches]))
    if matches:return matches[0]
    if RESUME_RUN_ROOT is not None or rejected:
        raise ValueError('No compatible resume folder. No new run was started. Check settings/package versions: '+str(rejected))
    digest=canonical_hash(expected)[:16]
    return OUTPUT_BASE/PROFILE/digest/f'shards_{N_SHARDS}',expected


In [ ]:
SCIENTIFIC=dict(analysis_version=ANALYSIS_VERSION,assays=ASSAYS,profile=PROFILE,
    min_paired_n=MIN_PAIRED_N,min_plexes=MIN_PLEXES,feature_scope=FEATURE_SCOPE,targets=SENSITIVITY_TARGETS,
    sensitivities=RUN_SENSITIVITIES,categorical=RUN_CATEGORICAL,influence=RUN_INFLUENCE_REFITS,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,compatible_log2=COMPATIBLE_LOG2_RATIO,scale_note=SCALE_PROVENANCE_NOTE,
    config={k:v for k,v in CONFIG.items() if k!='CORES'})
FUNCTIONS=['sha256_file','canonical_hash','write_json','write_csv','truth','unpack_inputs','load_signature_links','load_data','make_design','prepare_features','feature_cohort',
    'variant_spec','plan_jobs','build_model','posterior_array','hdi','effect_summary','numerical_diagnostics',
    'compute_ppc','compute_loo','sample_model','fit_one','run_deletions']
# Later bootstrap/aggregation functions get a second code hash in the final initialization cell.


In [ ]:
def linear_benchmark(d,Y):
    groups=pd.Categorical(d.plex).codes
    cov=np.column_stack([d.age.to_numpy()/10,d.sex_centered.to_numpy()])
    dummy=np.eye(groups.max()+1)[groups]
    bmi=d.bmi.to_numpy()/5
    def coefficient(y,extra=None):
        nuisance=np.column_stack([dummy,cov]+([] if extra is None else [extra]))
        resid=bmi-nuisance@np.linalg.lstsq(nuisance,bmi,rcond=None)[0]
        den=resid@resid
        return float(resid@y/den) if den>1e-8 else np.nan
    return np.array([coefficient(Y[:,0]),coefficient(Y[:,1]),coefficient(Y[:,1],Y[:,0])])

def bootstrap_targets(run):
    if PROFILE=='smoke':return
    targets=FEATURES[FEATURES.target];rng=np.random.default_rng(CONFIG['SEED']+999)
    group=DATA['batches']['protein'];parts=[np.flatnonzero(group==g) for g in sorted(set(group))]
    resamples=[np.concatenate([rng.choice(ix,len(ix),replace=True) for ix in parts]) for _ in range(BOOTSTRAP_REPLICATES)]
    rows=[]
    for f in targets.to_dict('records'):
        a=ARRAYS[f['key']];d=feature_cohort(DATA,f['assay']);samples=[]
        for ix in resamples:
            ix=ix[a['mask'][ix]]
            samples.append(linear_benchmark(d.iloc[ix].reset_index(drop=True),a['Y'][ix]))
        samples=np.asarray(samples);point=linear_benchmark(d.loc[a['mask']].reset_index(drop=True),a['Y'][a['mask']])
        for j,quantity in enumerate(['Protein','PTM','PTM_given_Protein']):
            v=samples[:,j];v=v[np.isfinite(v)]
            lo,hi=np.quantile(v,[.025,.975]) if len(v)>=100 else (np.nan,np.nan)
            rows.append(dict(key=f['key'],feature_id=f['feature_id'],assay=f['assay'],quantity=quantity,
                estimate=point[j],ci_95_lower=lo,ci_95_upper=hi,valid_replicates=len(v),requested=BOOTSTRAP_REPLICATES,
                interval_type='percentile 95% CI, fixed-plex linear benchmark'))
    write_csv(run/'bootstrap_target_linear_benchmark.csv',rows)


In [ ]:
def aggregate_outputs(root,run,contract):
    effects=[];status=[];ppcs=[];influence=[];issues=[]
    for shard in range(N_SHARDS):
        src=root/f'shard_{shard:02d}'
        if not (src/'shard_complete.json').exists():raise RuntimeError(f'Shard {shard} incomplete')
        if json.loads((src/'contract.json').read_text())!=contract:raise RuntimeError('Shard contract mismatch')
        expected=JOBS[JOBS.shard.eq(shard)]
        marker=json.loads((src/'shard_complete.json').read_text())
        if marker['jobs']!=expected[['key','variant']].to_dict('records'):raise RuntimeError('Shard ownership mismatch')
        for job in expected.to_dict('records'):
            folder=src/'fits'/job['key']/job['variant'];s=json.loads((folder/'status.json').read_text())
            if not s.get('postprocessing_complete'):raise RuntimeError('Incomplete fit '+str(folder))
            status.append(s);base=pd.read_csv(folder/'effects_95HDI.csv');effects.append(base)
            ppcs.append(pd.read_csv(folder/'predictive_checks.csv').assign(key=job['key'],variant=job['variant']))
            if job['variant']=='primary':
                psis=pd.read_csv(folder/'PSIS_LOO_patients.csv')
                for pid in psis.loc[truth(psis.flag_for_refit),'base_sample_id']:
                    deleted=folder.parent/('primary__without_'+pid)
                    state=deleted/'status.json'
                    adequate=state.exists() and json.loads(state.read_text()).get('postprocessing_complete',False) and json.loads(state.read_text()).get('numerical_checks_pass',False)
                    issues.append(dict(key=job['key'],patient=pid,deletion_numerically_adequate=adequate))
                    if adequate:
                        dt=pd.read_csv(deleted/'effects_95HDI.csv')
                        comparison=base.merge(dt,on=['key','quantity','term'],suffixes=('_primary','_deletion'))
                        comparison['deleted_patient']=pid
                        comparison['posterior_mean_change']=comparison.posterior_mean_deletion-comparison.posterior_mean_primary
                        influence.append(comparison)
    result=pd.concat(effects,ignore_index=True)
    if result.duplicated(['key','variant','quantity','term']).any():raise ValueError('Duplicated result rows')
    write_csv(run/'all_effects_95HDI.csv',result);write_csv(run/'fit_status.csv',status)
    write_csv(run/'predictive_checks.csv',pd.concat(ppcs,ignore_index=True))
    write_csv(run/'influence_status.csv',issues)
    if influence:write_csv(run/'patient_deletion_changes.csv',pd.concat(influence,ignore_index=True))
    if len(LINKS):
        # Expanded label table is annotation, not independent observations or a discovery count.
        write_csv(run/'primary_effects_linked_to_914_labels.csv',result[result.variant.eq('primary')].merge(LINKS,on='gene',how='inner'))
    summary=dict(planned_fits=len(JOBS),finished_fits=len(status),numerical_pass=sum(s['numerical_checks_pass'] for s in status),
        predictive_flags=sum(s['predictive_flags'] for s in status),psis_flagged_pairs=len(issues),
        unresolved_deletion_checks=sum(not x['deletion_numerically_adequate'] for x in issues),
        reporting='Pointwise 95% HDIs only; no ROPE or panel-wide FDR control.',
        influence_note='Adequate deletion refits require effect-change review; they do not repair PSIS scores.',
        calibration='No PTM interval-calibration simulation completed by this notebook.',
        annotation_policy='Source processed, localization probabilities unavailable' if SITE_QC_CSV is None else 'External QC table',
        scope='Bulk gene-level parent protein paired with individual PTM features; no causal or occupancy claim')
    write_json(run/'completion_summary.json',summary)
    return result,summary

def plot_and_package(result,run):
    import matplotlib.pyplot as plt
    primary=result[(result.variant=='primary')&result.numerical_checks_pass]
    fig,axs=plt.subplots(1,len(ASSAYS),figsize=(6*len(ASSAYS),5),squeeze=False)
    for ax,assay in zip(axs[0],ASSAYS):
        w=primary[primary.assay.eq(assay)].pivot(index='key',columns='quantity',values='posterior_mean')
        if {'PTM','PTM_given_Protein'}<=set(w):ax.scatter(w.PTM,w.PTM_given_Protein,s=8,alpha=.4)
        ax.axhline(0,color='grey',lw=.6);ax.axvline(0,color='grey',lw=.6)
        ax.set(title=assay,xlabel='Marginal PTM BMI slope',ylabel='Protein-conditioned PTM BMI slope')
    fig.suptitle('Standardized response per 5 kg/m²; point estimates, no discovery classification')
    fig.tight_layout();fig.savefig(run/'PTM_conditional_comparison.png',dpi=180);plt.show()
    targetkeys=set(FEATURES.loc[FEATURES.target,'key'])
    display(primary[primary.key.isin(targetkeys)&primary.quantity.eq('PTM_given_Protein')][
        ['assay','feature_id','posterior_mean','hdi_95_lower','hdi_95_upper']])
    dest=run/'Protein_PTM_reports.zip';tmp=dest.with_suffix('.partial.zip')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(run.iterdir()):
            if p.is_file() and p not in (dest,tmp):z.write(p,p.name)
    os.replace(tmp,dest);return dest


In [ ]:
FUNCTIONS+=['linear_benchmark','bootstrap_targets','aggregate_outputs','plot_and_package']
legacy_namespace=dict(globals())
exec(LEGACY_FUNCTION_SOURCE,legacy_namespace)
LEGACY_CODE_HASH=legacy_namespace['fingerprint_analysis'](FUNCTIONS)
CODE_HASH=fingerprint_analysis(FUNCTIONS+['sample_checkpointed','load_saved_trace','checkpoint_complete','save_job_progress','choose_resume_root'])
CONTRACT=dict(scientific=SCIENTIFIC,input_zip_sha256=INPUT_HASH,
    signature_zip_sha256=sha256_file(SIGNATURE_INPUT_ZIP) if SIGNATURE_INPUT_ZIP.exists() else None,
    qc_sha256=sha256_file(Path(SITE_QC_CSV)) if SITE_QC_CSV else None,code_hash=CODE_HASH,
    versions={p:importlib.metadata.version(p) for p in ['pymc','arviz','numpy','pandas','pytensor','scipy']},
    features=FEATURES.key.tolist(),jobs=JOBS.to_dict('records'))
RUN_ROOT,CONTRACT=choose_resume_root(CONTRACT,LEGACY_CODE_HASH)
DIGEST=RUN_ROOT.parent.name
RUN_DIR=RUN_ROOT/('aggregate' if MODE=='aggregate' else f'shard_{SHARD_ID:02d}' if MODE=='worker' else f'preflight_{SHARD_ID:02d}')
RUN_DIR.mkdir(parents=True,exist_ok=True)
if (RUN_DIR/'contract.json').exists() and json.loads((RUN_DIR/'contract.json').read_text())!=CONTRACT:raise ValueError('Refusing to overwrite a different run contract')
write_json(RUN_DIR/'resume_patch.json',{'patch':'batch_checkpoint_v1','execution_code_hash':CODE_HASH,'time':time.time()})
write_json(RUN_DIR/'contract.json',CONTRACT);write_json(RUN_DIR/'signature_status.json',SIGNATURE_STATUS)
write_csv(RUN_DIR/'eligible_features.csv',FEATURES);write_csv(RUN_DIR/'coverage_exclusions.csv',ELIGIBILITY)
write_csv(RUN_DIR/'planned_jobs.csv',JOBS);write_csv(RUN_DIR/'unidentifiable_variants.csv',UNIDENTIFIABLE)
write_csv(RUN_DIR/'cohort_covariates.csv',DATA['cohort'])
if len(LINKS):write_csv(RUN_DIR/'site_signature_mapping.csv',FEATURES.merge(LINKS,on='gene',how='inner'))
print('Output:',RUN_DIR)
if MODE=='preflight':print('No model sampling requested. Inspect eligibility and unresolved annotation/scale provenance before full estimation.')

if MODE=='worker':
    progress=save_job_progress(RUN_DIR)
    print('Saved job states:',progress.state.value_counts().to_dict())


In [ ]:
if MODE=='worker':
    owned=JOBS[JOBS.shard.eq(SHARD_ID)]
    for assay in ASSAYS:
        first=owned[(owned.assay==assay)&(owned.variant=='primary')].head(1)
        if len(first) and not (RUN_DIR/f'{assay}_prior_predictive.json').exists():
            key=first.iloc[0]['key'];a=ARRAYS[key]
            des=make_design(feature_cohort(DATA,assay),a['mask'])
            model=build_model(des,a['Y'][a['mask']],CONFIG)
            with model: prior=pm.sample_prior_predictive(samples=200,random_seed=CONFIG['SEED'])
            draws=prior.prior_predictive.paired_score.values
            if not np.isfinite(draws).all():raise ValueError('Non-finite prior predictive draws')
            write_json(RUN_DIR/f'{assay}_prior_predictive.json',{'absolute_quantiles_50_95_99':np.quantile(np.abs(draws),[.5,.95,.99]).tolist(),'note':'Scale check; not interval calibration'})
    fmap=FEATURES.set_index('key').to_dict('index')
    for count,job in enumerate(owned.to_dict('records'),1):
        feature={'key':job['key'],**fmap[job['key']]}
        try:
            print(f'Job {count}/{len(owned)}')
            fit_one(feature,job['variant'],RUN_DIR)
            if job['variant']=='primary':run_deletions(feature,RUN_DIR)
        except BaseException:
            (RUN_DIR/'last_error.txt').write_text(traceback.format_exc())
            save_job_progress(RUN_DIR)
            raise
        finally:
            save_job_progress(RUN_DIR)
    write_json(RUN_DIR/'shard_complete.json',{'jobs':owned[['key','variant']].to_dict('records'),'run':DIGEST})
    print('Shard finished. Numerical and predictive adequacy are assessed separately.')


In [ ]:
if MODE=='aggregate':
    RESULTS,SUMMARY=aggregate_outputs(RUN_ROOT,RUN_DIR,CONTRACT)
    bootstrap_targets(RUN_DIR)
    REPORT_ZIP=plot_and_package(RESULTS,RUN_DIR)
    display(SUMMARY);print('Report ZIP:',REPORT_ZIP)
    # Optional local download:
    # from google.colab import files
    # files.download(str(REPORT_ZIP))
